In [1]:
pip install -U transformers datasets peft trl accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 16.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 65.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 31.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 59.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 82.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 81.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 27.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 81.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [peft]2m21/22 [peft]formers]ub]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may nee

In [6]:
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto"
)

print("Loading datasets...")

train_dataset = load_dataset(
    "json",
    data_files="train_qwen.jsonl",
    split="train"
)

eval_dataset = load_dataset(
    "json",
    data_files="eval_qwen.jsonl",
    split="train"
)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print("Applying LoRA...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_java_security",

    num_train_epochs=5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    bf16=True,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    report_to="none"
)

trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    processing_class=tokenizer
)

print("Starting training...")

trainer.train()

print("Saving model...")

trainer.save_model(
    "./qwen_java_security_final"
)

tokenizer.save_pretrained(
    "./qwen_java_security_final"
)

print("Training complete")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loading datasets...
Train: 1202
Eval : 213
Applying LoRA...


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637


/opt/venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.029897,0.029846,0.030202,0.988243,752587.000000
100,0.026142,0.026466,0.026838,0.989409,1494979.000000
150,0.026033,0.025088,0.025739,0.989612,2246565.000000
200,0.021853,0.024437,0.023569,0.989697,2987531.000000
250,0.020729,0.023778,0.022294,0.990144,3726844.000000
300,0.020754,0.023211,0.022009,0.990295,4477853.000000
350,0.019721,0.023268,0.021338,0.990299,5221330.000000
380,0.020855,0.023259,0.021336,0.990375,5656370.000000


Saving model...
Training complete


In [38]:
from datasets import load_dataset
 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
 
from peft import (
    LoraConfig,
    get_peft_model
)
 
from trl import SFTTrainer
 
MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"
 
print("Loading tokenizer...")
 
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
 
tokenizer.pad_token = tokenizer.eos_token
 
print("Loading model...")
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype="auto"
)
 
print("Loading datasets...")
 
train_dataset = load_dataset(
    "json",
    data_files="train_detect_fix_clean.jsonl",
    split="train"
)
 
eval_dataset = load_dataset(
    "json",
    data_files="eval_detect_fix_clean.jsonl",
    split="train"
)
 
print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))
 
print("Applying LoRA...")
 
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)
 
model = get_peft_model(
    model,
    lora_config
)
 
model.print_trainable_parameters()
 
training_args = TrainingArguments(
    output_dir="./qwen_security_lora",
 
    num_train_epochs=3,
 
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
 
    gradient_accumulation_steps=16,
 
    learning_rate=2e-4,
 
    warmup_steps=50,
 
    lr_scheduler_type="cosine",
 
    bf16=True,
 
    logging_steps=10,
 
    eval_strategy="steps",
    eval_steps=50,
 
    save_strategy="steps",
    save_steps=100,
 
    save_total_limit=2,
 
    report_to="none"
)
 
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)
 
print("Starting training...")
 
trainer.train()
 
print("Saving LoRA adapter...")
 
trainer.save_model(
    "./qwen_security_lora_final"
)
 
tokenizer.save_pretrained(
    "./qwen_security_lora_final"
)
 
print("Training Complete")
 

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Train: 1202
Eval : 213
Applying LoRA...
trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637


...

Tokenizing train dataset:   0%|          | 0/1202 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.030650,0.029648,0.031511,0.989703,714330.000000
100,0.023204,0.023932,0.023909,0.991094,1419491.000000
150,0.023035,0.021913,0.023045,0.991476,2132689.000000
200,0.018036,0.020967,0.020800,0.991671,2835949.000000
228,0.020431,0.020847,0.020751,0.991805,3221472.000000


Saving LoRA adapter...
Training Complete


In [2]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import PeftModel
 
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-14B-Instruct",
    torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-Coder-14B-Instruct",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base_model,
    "./qwen_security_lora_final"
)
 
merged_model = model.merge_and_unload()
 

 

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
merged_model.save_pretrained(
    "../../qwen_security_merged"
)
 
tokenizer.save_pretrained(
    "../../qwen_security_merged"
)
 
print("Merged model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
 
MODEL_PATH = "../../qwen_security_merged"
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
 
java_code = open("../../sample.java").read()
 

In [ ]:
prompt = f"""
### Task
Analyze the Java code and fix the vulnerability
 
### Java Code
 
{java_code}
 
### Vulnerability
"""
 
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)
 
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=False
)

result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
print(
    result.split("### Vulnerability")[-1].strip()
)

In [ ]:
streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false

In [9]:
parts = result.split("### Vulnerability")[-1].strip().split("### Secure Code")

print(parts[1])



import java.sql.*;

public class UserLogin {
    public void loginUser(Connection connection, String username, String password) throws SQLException {
        // SECURE: Use parameterized queries to prevent SQL Injection
        String query = "SELECT * FROM users WHERE username = ? AND password = ?";
        PreparedStatement preparedStatement = connection.prepareStatement(query);
        preparedStatement.setString(1, username);
        preparedStatement.setString(2, password);
        ResultSet resultSet = preparedStatement.executeQuery();
        
        if (resultSet.next()) {
            System.out.println("Login successful!");
        }
    }
}
